In [16]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
RESULTS = ROOT / "reports" / "results"
FIGURES = ROOT / "reports" / "figures"
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
from src.data import load_raw, dedupe, temporal_split, random_split
from src.evaluate import fit_eval, recall_at_precision, precision_at_k

df = load_raw()

In [17]:
FEATURES = [c for c in df.columns if c not in ("Class",)]

cdf, n_dupes = dedupe(df)
print(f"Removed {n_dupes} duplicate rows.")


Removed 1081 duplicate rows.


In [18]:
SEED = 42
results, curves = {}, {}

X_res, y_res = SMOTE(random_state=SEED).fit_resample(cdf[FEATURES], cdf["Class"])
leaked = pd.concat([X_res, y_res], axis=1)
tra, tea = random_split(leaked)
results["A"], ya, sa = fit_eval(tra, tea, options="none", feature=FEATURES, seed=SEED)
curves["A"] = precision_recall_curve(ya, sa)

trb, teb = random_split(cdf)
results["B"], yb, sb = fit_eval(trb, teb, options="smote", feature=FEATURES, seed=SEED)
curves["B"] = precision_recall_curve(yb, sb)

trt, cbt ,tet = temporal_split(cdf)
results["C"], yc, sc = fit_eval(trt, tet, options="smote", feature=FEATURES, seed=SEED)
curves["C"] = precision_recall_curve(yc, sc)

results["D"], yd, sd = fit_eval(trt, tet, options="weights", feature=FEATURES, seed=SEED)
curves["D"] = precision_recall_curve(yd, sd)

TypeError: fit_eval() got an unexpected keyword argument 'options'

In [ ]:
LABELS = {
    "A": "Random split, SMOTE before split",
    "B": "Random split, SMOTE inside pipeline",
    "C": "Temporal split, SMOTE inside pipeline",
    "D": "Temporal split, class weights",
}
 
table = (
    pd.DataFrame(results).T
      .rename_axis("config")
      .reset_index()
      .assign(label=lambda d: d["config"].map(LABELS))
)
table["pr_auc_inflation_vs_D"] = table["pr_auc"] - table.loc[
    table["config"] == "D", "pr_auc"
].iloc[0]
 
cols = ["config", "label", "pr_auc", "pr_auc_inflation_vs_D", "roc_auc",
        "recall_at_p90", "n_train", "n_test", "n_pos_test",
        "prevalence_test", "synthetic_share_of_test_pos"]
table = table[cols]
 
table.to_csv(RESULTS / "leakage_experiment.csv", index=False)
np.savez_compressed(
    RESULTS / "leakage_curves.npz",
    **{f"{k}_{name}": arr
       for k, (prec, rec, _) in curves.items()
       for name, arr in (("precision", prec), ("recall", rec))},
)
 
display(table.round(4))